# EXTRACT AUDIO FILES CATEGORICALLY

In [1]:
import pandas as pd
import os
import subprocess
import numpy as np
from collections import defaultdict
import shutil
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# 1. CONFIGURATION
# =====================================================
EXCEL_FILE = "C:\\FINAL DATASET\\train_lable.xlsx"
OUTPUT_DIR = "C:\\FINAL DATASET\\extracted_files_2"
SOURCE_COVER = "C:\\FINAL DATASET\\g729a_0"
SOURCE_STEGO = "C:\\FINAL DATASET\\g729a_Steg"

TOTAL_PAIRS_NEEDED = 50000 
TARGET_01_RATE = 30000      
SILENCE_THRESHOLD = -45.0 

TABS = ["train_lable_english_CNV", "train_lable_english_PMS", 
        "train_lable_chinese_CNV", "train_lable_chinese_PMS"]

# =====================================================
# 2. CORE FUNCTIONS
# =====================================================

def check_file_db(file_info):
    """Worker function for parallel processing"""
    fname, c_src, s_src, c_target, s_target = file_info
    
    cmd = [
        'ffmpeg', '-y', '-f', 'g729', '-i', c_src,
        '-f', 's16le', '-acodec', 'pcm_s16le', '-ar', '8000', '-ac', '1',
        '-loglevel', 'quiet', '-'
    ]
    try:
        # We use a short timeout to prevent FFmpeg from hanging on corrupted files
        raw_audio = subprocess.check_output(cmd, timeout=10)
        audio_array = np.frombuffer(raw_audio, dtype=np.int16).astype(np.float32) / 32768.0
        if len(audio_array) == 0: return None
        
        rms = np.sqrt(np.mean(audio_array**2))
        db = 20 * np.log10(rms) if rms > 0 else -float('inf')
        
        if db >= SILENCE_THRESHOLD:
            return (c_src, s_src, c_target, s_target)
    except:
        pass
    return None

def build_file_index(root_dir):
    print(f"Indexing files in {root_dir}...")
    index = {}
    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if f.endswith('.g729a'):
                index[f] = os.path.join(dirpath, f)
    return index

def load_and_organize_metadata(excel_file):
    print(f"Loading metadata from {excel_file}...")
    organized = defaultdict(list)
    xls = pd.ExcelFile(excel_file)
    for tab in TABS:
        df = pd.read_excel(xls, sheet_name=tab)
        parts = tab.replace("train_lable_", "").split("_")
        lang, algo = parts[0], parts[1]
        for _, row in df.iterrows():
            if pd.notna(row.iloc[0]) and pd.notna(row.iloc[2]):
                fname = str(row.iloc[0]).strip() + ".g729a"
                rate = round(float(row.iloc[2]), 1)
                organized[(lang, algo, rate)].append(fname)
    return organized

def calculate_weighted_distribution(meta_groups):
    rate_groups = defaultdict(list)
    for key in meta_groups.keys():
        rate_groups[key[2]].append(key)
    
    distribution = {}
    keys_01 = rate_groups.get(0.1, [])
    if keys_01:
        per_key_01 = TARGET_01_RATE // len(keys_01)
        for k in keys_01: distribution[k] = per_key_01
    
    remaining_pairs = TOTAL_PAIRS_NEEDED - TARGET_01_RATE
    other_keys = [k for r, keys in rate_groups.items() if r != 0.1 for k in keys]
    if other_keys:
        per_key_other = remaining_pairs // len(other_keys)
        for k in other_keys: distribution[k] = per_key_other
    return distribution

# =====================================================
# 3. EXECUTION PIPELINE
# =====================================================

def main():
    print(f"{'='*60}\nAUSPEX: High-Stability Threaded Extraction\n{'='*60}")

    cover_index = build_file_index(SOURCE_COVER)
    stego_index = build_file_index(SOURCE_STEGO)
    meta_groups = load_and_organize_metadata(EXCEL_FILE)
    targets = calculate_weighted_distribution(meta_groups)

    cover_out = os.path.join(OUTPUT_DIR, "cover")
    stego_out = os.path.join(OUTPUT_DIR, "stego")
    os.makedirs(cover_out, exist_ok=True)
    os.makedirs(stego_out, exist_ok=True)

    overall_total_pairs = 0

    for category, target_count in targets.items():
        lang, algo, rate = category
        filenames = sorted(meta_groups[category])
        print(f"\nProcessing {lang.upper()} | {algo} | Rate {rate} (Target: {target_count})")
        
        cat_count = 0
        pbar = tqdm(total=target_count, desc="Progress")

        # --- STEP 1: COUNT EXISTING VALID FILES ---
        to_process = []
        for fname in filenames:
            c_target = os.path.join(cover_out, fname)
            s_target = os.path.join(stego_out, fname)
            
            if os.path.exists(c_target) and os.path.exists(s_target):
                if cat_count < target_count:
                    cat_count += 1
                    overall_total_pairs += 1
                    pbar.update(1)
            else:
                c_src = cover_index.get(fname)
                s_src = stego_index.get(fname)
                if c_src and s_src:
                    to_process.append((fname, c_src, s_src, c_target, s_target))

        # --- STEP 2: MULTI-THREADED FFMEG & COPY ---
        if cat_count < target_count and to_process:
            print(f"--> Starting FFmpeg analysis for {len(to_process)} missing files...")
            
            # Reduced to 4 workers to prevent HDD "Thrashing"
            with ThreadPoolExecutor(max_workers=12) as executor:
                future_to_file = {executor.submit(check_file_db, item): item for item in to_process}
                
                for future in as_completed(future_to_file):
                    if cat_count >= target_count:
                        break
                    
                    try:
                        result = future.result() # This will wait here if FFmpeg is still running
                        if result:
                            c_src, s_src, c_target, s_target = result
                            shutil.copy2(c_src, c_target)
                            shutil.copy2(s_src, s_target)
                            cat_count += 1
                            overall_total_pairs += 1
                            pbar.update(1)
                        else:
                            # If a file is skipped due to silence, we still need to 
                            # know the script is alive, but we don't update cat_count
                            pass 
                    except Exception as e:
                        print(f"\nError processing a file: {e}")
                        continue
        
        pbar.close()

    print(f"\n{'='*60}\nProcess Complete: {overall_total_pairs} pairs saved.\n{'='*60}")

if __name__ == "__main__":
    main()

AUSPEX: High-Stability Threaded Extraction
Indexing files in C:\FINAL DATASET\g729a_0...
Indexing files in C:\FINAL DATASET\g729a_Steg...
Loading metadata from C:\FINAL DATASET\train_lable.xlsx...

Processing ENGLISH | CNV | Rate 0.1 (Target: 7500)


Progress: 100%|██████████| 7500/7500 [00:02<00:00, 3436.07it/s]



Processing ENGLISH | PMS | Rate 0.1 (Target: 7500)


Progress: 100%|██████████| 7500/7500 [00:02<00:00, 3723.75it/s]



Processing CHINESE | CNV | Rate 0.1 (Target: 7500)


...


Processing CHINESE | CNV | Rate 0.4 (Target: 1666)


Progress: 100%|██████████| 1666/1666 [00:01<00:00, 1616.55it/s]



Processing CHINESE | PMS | Rate 0.4 (Target: 1666)


Progress: 100%|██████████| 1666/1666 [00:00<00:00, 1915.79it/s]


Process Complete: 49992 pairs saved.


# VERIFY AUDIO

In [3]:
import os
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

# =====================================================
# CONFIGURATION
# =====================================================
EXCEL_FILE = "C:\\FINAL DATASET\\train_lable.xlsx"
EXTRACTED_DIR = "C:\\FINAL DATASET\\extracted_files"
SOURCE_COVER = "C:\\FINAL DATASET\\g729a_0"
SOURCE_STEGO = "C:\\FINAL DATASET\\g729a_Steg"

TABS = ["train_lable_english_CNV", "train_lable_english_PMS", 
        "train_lable_chinese_CNV", "train_lable_chinese_PMS"]

TOTAL_PAIRS_NEEDED = 50000
TARGET_01_RATE = 30000

def get_source_sizes(root):
    size_map = {}
    print(f"Indexing source sizes in {root}...")
    for dp, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith('.g729a'):
                size_map[f] = os.path.getsize(os.path.join(dp, f))
    return size_map

def run_sync_audit():
    print(f"{'='*60}\nAUSPEX: Dataset Audit & Sync (Size Validation)\n{'='*60}")
    
    # 1. Map Source Sizes for Integrity Check
    src_c_sizes = get_source_sizes(SOURCE_COVER)
    src_s_sizes = get_source_sizes(SOURCE_STEGO)

    # 2. Re-calculate exactly which files SHOULD be there
    organized = defaultdict(list)
    xls = pd.ExcelFile(EXCEL_FILE)
    for tab in TABS:
        df = pd.read_excel(xls, sheet_name=tab)
        lang, algo = tab.replace("train_lable_", "").split("_")[0:2]
        for _, row in df.iterrows():
            if pd.notna(row.iloc[0]) and pd.notna(row.iloc[2]):
                fname = str(row.iloc[0]).strip() + ".g729a"
                rate = round(float(row.iloc[2]), 1)
                organized[(lang, algo, rate)].append(fname)

    rate_groups = defaultdict(list)
    for key in organized.keys():
        rate_groups[key[2]].append(key)
    
    intended_files = set()
    
    # Logic for 0.1 rate
    keys_01 = rate_groups.get(0.1, [])
    per_key_01 = TARGET_01_RATE // len(keys_01)
    for k in keys_01:
        intended_files.update(sorted(organized[k])[:per_key_01])

    # Logic for 0.2, 0.3, 0.4 rates
    remaining_pairs = TOTAL_PAIRS_NEEDED - TARGET_01_RATE
    other_keys = [k for r, keys in rate_groups.items() if r != 0.1 for k in keys]
    per_key_other = remaining_pairs // len(other_keys)
    for k in other_keys:
        intended_files.update(sorted(organized[k])[:per_key_other])

    # 3. Audit Folders
    deleted_count = 0
    mismatch_count = 0
    
    for folder in ['cover', 'stego']:
        folder_path = os.path.join(EXTRACTED_DIR, folder)
        if not os.path.exists(folder_path): continue
        
        current_files = os.listdir(folder_path)
        print(f"\nAuditing {folder.upper()} folder...")
        
        for f in tqdm(current_files):
            full_path = os.path.join(folder_path, f)
            
            # --- Check 1: Is it a "Ghost File" (Not in intended set)? ---
            if f not in intended_files:
                # os.remove(full_path)
                deleted_count += 1
                continue
            
            # --- Check 2: Size Integrity (Was it corrupted by power loss)? ---
            ref_size = src_c_sizes.get(f) if folder == 'cover' else src_s_sizes.get(f)
            if ref_size and os.path.getsize(full_path) != ref_size:
                # os.remove(full_path)
                mismatch_count += 1

    print(f"\n{'='*60}")
    print(f"Sync Complete!")
    print(f"Deleted Ghost Files: {deleted_count}")
    print(f"Deleted Corrupted (Size Mismatch): {mismatch_count}")
    print(f"Intended Final Count: {len(intended_files)} pairs")
    print(f"Action: You can now re-run your main script to fill the missing slots.")
    print(f"{'='*60}")

if __name__ == "__main__":
    run_sync_audit()

AUSPEX: Dataset Audit & Sync (Size Validation)
Indexing source sizes in C:\FINAL DATASET\g729a_0...
Indexing source sizes in C:\FINAL DATASET\g729a_Steg...

Auditing COVER folder...


100%|██████████| 34037/34037 [00:02<00:00, 15020.17it/s]



Auditing STEGO folder...


100%|██████████| 34037/34037 [00:02<00:00, 15696.29it/s]


Sync Complete!
Deleted Ghost Files: 5750
Deleted Corrupted (Size Mismatch): 0
Intended Final Count: 49992 pairs
Action: You can now re-run your main script to fill the missing slots.
